# Patch SAE segmentation: run on Kaggle and download results

Enable **Internet** and select **T4 GPU**. No Hugging Face account, secret,
token, or artifact repository is needed. Public pretrained models and datasets
are downloaded automatically; experiment artifacts stay in Kaggle.

Push the updated code to GitHub `refactor` before running. Start with one
encoder/dataset job: 3 SAE fits and 4 evaluations. The full six-job campaign
has 18 fits and 24 evaluations, all sparse inference at K=12.

After each completed evaluation, a small results ZIP is refreshed automatically.
Download it before ending the session. Local checkpoints support resume only
while the workspace is retained; the results ZIP does not contain model weights.


In [ ]:
ENCODER = "dinov2"  # "dinov2", "siglip2", "clip"; "all" runs sequentially
DATASET = "voc"     # "voc", "ade20k"; "all" runs sequentially
GIT_REPO = "https://github.com/thienquang204/MPSAE.git"
GIT_BRANCH = "refactor"


In [ ]:
import os, sys, subprocess, json
from pathlib import Path

# Public downloads only. No login or secret lookup.
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
os.environ["HF_HOME"] = "/tmp/mpsae-hf"
os.environ["HF_DATASETS_CACHE"] = "/tmp/mpsae-datasets"
os.environ["PYTHONUNBUFFERED"] = "1"
os.environ["MPLBACKEND"] = "Agg"
REPO = Path("/kaggle/working/MPSAE")
WORK = Path("/kaggle/working/patch-scene-v2")
if not REPO.exists():
    subprocess.run(["git", "clone", "--branch", GIT_BRANCH, "--depth", "1", GIT_REPO, str(REPO)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
os.chdir(REPO)
subprocess.run(["git", "rev-parse", "HEAD"], check=True)
subprocess.run(["nvidia-smi"], check=True)
assert (REPO / "mpsae/patch_scene/__main__.py").exists(), "Push the new code to GitHub first."
assert "def export_results" in (REPO / "mpsae/patch_scene/report.py").read_text(), "Push the local-results update first."


## Isolated environment
Uses Python 3.11 and CUDA 12.6 PyTorch, independently of Kaggle's installed
Python/packages. P100 is unsupported by the project guard. T4 x2 uses GPU 0.


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
PYTHON = "/tmp/mpsae-env/bin/python"
if not Path(PYTHON).exists():
    subprocess.run([sys.executable, "-m", "uv", "venv", "--python", "3.11", "--seed", "/tmp/mpsae-env"], check=True)
subprocess.run([PYTHON, "-m", "pip", "install", "--no-cache-dir",
                "torch==2.9.1", "torchvision==0.24.1",
                "--index-url", "https://download.pytorch.org/whl/cu126"], check=True)
subprocess.run([PYTHON, "-m", "pip", "install", "--no-cache-dir",
                "-r", "requirements-patch-scene.txt"], check=True)
subprocess.run([PYTHON, "-m", "pip", "check"], check=True)
subprocess.run([PYTHON, "-m", "mpsae.patch_scene", "doctor", "--device", "cuda:0"], check=True)


In [ ]:
CONFIG = "configs/patch_scene_kaggle.yaml"
PLAN = json.loads(subprocess.check_output([
    PYTHON, "-m", "mpsae.patch_scene", "plan", "--config", CONFIG], text=True))
print(json.dumps(PLAN, indent=2))
assert PLAN["total_evaluations"] == 24 and PLAN["inference_k"] == [12]
print("Selected job:", DATASET, ENCODER)


## Run or resume locally
This cell trains/evaluates the selected job without logging in or uploading.
Running it again reuses completed work when the same local workspace remains.
Start with DINOv2/VOC. Change ENCODER/DATASET in the first cell for later jobs.

The automatic ZIP is at `/kaggle/working/patch-scene-v2/patch-scene-results.zip`.
If a later method fails, completed evaluations are still in that ZIP. On normal
exit or a Python exception it is also copied to `/kaggle/working/patch-scene-results.zip`.


In [ ]:
import shutil

command = [PYTHON, "-u", "-m", "mpsae.patch_scene", "run",
           "--config", CONFIG, "--encoder", ENCODER, "--dataset", DATASET,
           "--workspace", str(WORK), "--device", "cuda:0"]
try:
    subprocess.run(command, check=True)
finally:
    # Preserve already completed results even if a later method raises an error.
    saved = WORK / "patch-scene-results.zip"
    if saved.exists():
        shutil.copy2(saved, "/kaggle/working/patch-scene-results.zip")
        print("Download: /kaggle/working/patch-scene-results.zip")


In [ ]:
from IPython.display import FileLink, display

# This cell may also be run after interrupting the training cell.
# Export only combines completed local results; it does not train or evaluate.
subprocess.run([PYTHON, "-m", "mpsae.patch_scene", "export",
                "--config", CONFIG, "--workspace", str(WORK)], check=True)
archive = Path("/kaggle/working/patch-scene-results.zip")
import shutil
shutil.copy2(WORK / "patch-scene-results.zip", archive)
table = WORK / "campaigns" / PLAN["campaign"] / "table"
print((table / "results.tex").read_text())
# Use a relative link with the notebook working directory at /kaggle/working.
previous = Path.cwd()
try:
    os.chdir(archive.parent)
    display(FileLink(archive.name))
finally:
    os.chdir(previous)
print("Or use Kaggle's file browser: /kaggle/working/patch-scene-results.zip -> Download")


## Download and preserve
Download **patch-scene-results.zip** from the cell above or Kaggle's file browser
under `/kaggle/working`. It contains CSV, LaTeX, detailed JSON and run settings;
missing evaluations are `--`, never zero. No plots are produced.

There is no remote backup in this workflow. Download the ZIP before ending the
session, or save a notebook version with output files. Saving code alone does
not preserve experiment data. Checkpoints and caches remain in the workspace,
outside the small results ZIP. For another session, retaining/restoring the full
workspace is necessary to resume training; otherwise that job starts again.

See `docs/patch_scene_kaggle.md`. Optional HF backup flags remain available in
the CLI, but this notebook never enables them.
